In [ ]:
"""
Step 3: proper Granger causality test, fixing the shared-trend AND
shared-weekly-pattern problems found in the raw lag scan.

Workflow:
  1. DESEASONALIZE: regress each series on day-of-week dummies (+ a
     Patch-Tuesday indicator, since CVE disclosures cluster on the second
     Tuesday of the month) and keep the residuals. Differencing alone does
     NOT remove a repeating weekly pattern -- it only removes a straight-line
     trend -- so a shared weekly rhythm (Reddit posting dips on weekends;
     CVE disclosure clusters mid-week/Patch Tuesday) can survive the trend
     fix and still produce spurious "significance" in both directions.
  2. ADF unit-root test PLUS an explicit linear-trend test on the
     deseasonalized residuals. A series can pass the unit-root test while
     still having a real slope over time, so both checks run independently.
  3. If EITHER check flags a problem, difference BOTH series in a pair by
     the same order.
  4. VAR order selection (AIC and BIC) picks a defensible lag instead of
     scanning every lag and cherry-picking.
  5. Granger causality test at both selected lags, in BOTH directions:
       - does CVE volume Granger-cause burnout volume? (the hypothesis)
       - does burnout volume Granger-cause CVE volume? (should NOT be
         significant -- if it is, something shared is still contaminating
         both series)

Reads the CSV from q1_timeseries.py. No raw file access here.
"""

import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats as scipy_stats
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.tsa.api import VAR
import os
import warnings

warnings.filterwarnings("ignore")  # statsmodels VAR is chatty about lag-order edge cases

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
OUT_DIR = "/Users/nadia/Desktop/redditRun_june/q1_analysis/"
# DAILY_CSV = os.path.join(OUT_DIR, "q1_daily_series.csv")
DAILY_CSV = os.path.join(OUT_DIR, "q1_weekly_series.csv")
BAT_COLS = ["n_bat_ge1", "n_bat_ge2"]
CVE_COLS = ["n_cve_run1", "n_cve_run2"]

MAX_LAG_FOR_ORDER_SELECTION = 84
MAX_DIFF_ORDER = 2
ALPHA = 0.05


def is_patch_tuesday(date):
    """Second Tuesday of the month -- the day vendors (notably Microsoft)
    routinely release patches and disclose vulnerabilities, a well-known
    source of CVE-publication clustering."""
    return 1 if (date.weekday() == 1 and 8 <= date.day <= 14) else 0


def deseasonalize(daily_df, date_col, value_cols):
    """Regress each value column on day-of-week dummies + Patch-Tuesday
    indicator. Returns a DataFrame of residuals (same index/order as input)
    plus prints the R^2 for each column so you can see how much weekly
    pattern was actually present."""
    dates = pd.to_datetime(daily_df[date_col])
    dow = dates.dt.dayofweek
    patch_tues = dates.apply(is_patch_tuesday)

    dow_dummies = pd.get_dummies(dow, prefix="dow", drop_first=True)
    X = pd.concat([dow_dummies, patch_tues.rename("patch_tuesday")], axis=1)
    X = sm.add_constant(X).astype(float)

    residuals = {}
    print("  Day-of-week + Patch-Tuesday deseasonalization:")
    for col in value_cols:
        y = daily_df[col].astype(float)
        model = sm.OLS(y, X).fit()
        residuals[col] = model.resid
        print(f"    {col}: R^2 = {model.rsquared:.4f} "
              f"({'meaningful weekly pattern present' if model.rsquared > 0.02 else 'little/no weekly pattern'})")

    return pd.DataFrame(residuals)


def adf_pvalue(series):
    return adfuller(series.dropna(), autolag="AIC")[1]


def trend_pvalue_and_slope(series):
    """OLS regression of the series against a plain time index (0,1,2,...).
    Returns (p_value, slope) for the slope coefficient. A significant
    p-value means a real deterministic trend exists over the period, even
    if the ADF unit-root test alone doesn't flag it."""
    s = series.dropna().values
    t = np.arange(len(s))
    slope, intercept, r, p, se = scipy_stats.linregress(t, s)
    return p, slope


def find_common_diff_order(series_a, series_b, label_a, label_b, max_d=MAX_DIFF_ORDER):
    """Difference both series together, by the same number of steps, until
    BOTH pass the unit-root test AND show no significant linear trend (or
    until max_d is hit)."""
    a, b = series_a.copy(), series_b.copy()
    d = 0
    while d <= max_d:
        p_a_unit, p_b_unit = adf_pvalue(a), adf_pvalue(b)
        p_a_trend, slope_a = trend_pvalue_and_slope(a)
        p_b_trend, slope_b = trend_pvalue_and_slope(b)

        unit_a = "no unit root" if p_a_unit < ALPHA else "UNIT ROOT"
        unit_b = "no unit root" if p_b_unit < ALPHA else "UNIT ROOT"
        trend_a = f"TREND (slope={slope_a:.5f}, p={p_a_trend:.4g})" if p_a_trend < ALPHA else "no significant trend"
        trend_b = f"TREND (slope={slope_b:.5f}, p={p_b_trend:.4g})" if p_b_trend < ALPHA else "no significant trend"

        print(f"    d={d}: {label_a}: ADF p={p_a_unit:.4g} ({unit_a}); linear trend check: {trend_a}")
        print(f"           {label_b}: ADF p={p_b_unit:.4g} ({unit_b}); linear trend check: {trend_b}")

        a_ok = (p_a_unit < ALPHA) and (p_a_trend >= ALPHA)
        b_ok = (p_b_unit < ALPHA) and (p_b_trend >= ALPHA)

        if a_ok and b_ok:
            return a, b, d

        a = a.diff().dropna()
        b = b.diff().dropna()
        d += 1
    print(f"    WARNING: still non-stationary/trending after d={max_d} -- proceeding anyway, "
          f"interpret results cautiously")
    return a, b, d


def select_lag_orders(df_pair, maxlags):
    model = VAR(df_pair)
    sel = model.select_order(maxlags=maxlags)
    return sel


def granger_at_lag(effect_series, cause_series, lag, direction_label):
    """data columns must be [effect, cause] -- statsmodels tests whether
    column 1 (cause) Granger-causes column 0 (effect)."""
    df_pair = pd.DataFrame({"effect": effect_series.values, "cause": cause_series.values})
    results = grangercausalitytests(df_pair, maxlag=[lag], verbose=False)
    fstat, pvalue, df_denom, df_num = results[lag][0]["ssr_ftest"]
    sig = "SIGNIFICANT" if pvalue < ALPHA else "not significant"
    print(f"    {direction_label} at lag={lag}: F={fstat:.4f}, p={pvalue:.4g}  -> {sig} (alpha={ALPHA})")
    return fstat, pvalue


def main():
    if not os.path.exists(DAILY_CSV):
        print(f"Could not find {DAILY_CSV}. Run build_q1_timeseries.py first.")
        return

    daily = pd.read_csv(DAILY_CSV)
    print(f"Loaded {len(daily)} days from {DAILY_CSV}")

    print(f"\n{'=' * 70}")
    print("DESEASONALIZING (day-of-week + Patch Tuesday)")
    print("=" * 70)
    deseasonalized = deseasonalize(daily, "date", BAT_COLS + CVE_COLS)

    summary_rows = []

    for bat_col in BAT_COLS:
        for cve_col in CVE_COLS:
            print(f"\n{'=' * 70}")
            print(f"{bat_col}  vs  {cve_col}")
            print("=" * 70)

            print("\n  Stationarity check + common differencing order (on deseasonalized residuals):")
            bat_s, cve_s, d = find_common_diff_order(
                deseasonalized[bat_col], deseasonalized[cve_col], bat_col, cve_col
            )
            print(f"  Using d={d} for both series in this pair")

            df_pair = pd.DataFrame({bat_col: bat_s.values, cve_col: cve_s.values})

            print(f"\n  Selecting lag order (AIC/BIC, up to {MAX_LAG_FOR_ORDER_SELECTION} days):")
            sel = select_lag_orders(df_pair, MAX_LAG_FOR_ORDER_SELECTION)
            aic_lag = max(int(sel.aic), 1)
            bic_lag = max(int(sel.bic), 1)
            print(f"  AIC-selected lag: {aic_lag}   |   BIC-selected lag: {bic_lag}")

            print(f"\n  Granger causality -- does CVE lead burnout?")
            for lag, crit_name in [(aic_lag, "AIC"), (bic_lag, "BIC")]:
                fstat, pvalue = granger_at_lag(bat_s, cve_s, lag, f"CVE -> burnout ({crit_name} lag)")
                summary_rows.append({
                    "bat_threshold": bat_col, "cve_threshold": cve_col,
                    "diff_order": d, "criterion": crit_name, "lag": lag,
                    "direction": "cve_to_bat", "f_stat": round(fstat, 4), "p_value": round(pvalue, 6),
                })

            print(f"\n  Reverse check -- does burnout lead CVE? (should NOT be significant for a clean story)")
            for lag, crit_name in [(aic_lag, "AIC"), (bic_lag, "BIC")]:
                fstat, pvalue = granger_at_lag(cve_s, bat_s, lag, f"burnout -> CVE ({crit_name} lag)")
                summary_rows.append({
                    "bat_threshold": bat_col, "cve_threshold": cve_col,
                    "diff_order": d, "criterion": crit_name, "lag": lag,
                    "direction": "bat_to_cve", "f_stat": round(fstat, 4), "p_value": round(pvalue, 6),
                })

    summary_df = pd.DataFrame(summary_rows)
    out_path = os.path.join(OUT_DIR, "q1_granger_causality_results_v3(weekly).csv")
    summary_df.to_csv(out_path, index=False)

    print(f"\n{'=' * 70}")
    print("FULL SUMMARY")
    print("=" * 70)
    print(summary_df.to_string(index=False))
    print(f"\nSaved -> {out_path}")

    print(f"\n{'=' * 70}")
    print("HOW TO READ THIS")
    print("=" * 70)
    print("The bar for a real finding: direction='cve_to_bat' significant (p < 0.05)")
    print("AND the matching direction='bat_to_cve' row (same combination + criterion)")
    print("NOT significant. Only that combination counts as clean evidence that CVE")
    print("volume helps predict burnout volume, and not the other way around.")
    print("\nIf bat_to_cve is significant ANYWHERE, that specific row is not trustworthy")
    print("evidence for your hypothesis, regardless of what cve_to_bat shows for the same")
    print("combination -- it means something is still shared between the two series that")
    print("hasn't been fully removed (weekly pattern, trend, or some other confound).")
    print("\nWeight BIC rows more than AIC rows when they disagree -- AIC tends to pick")
    print("larger, more overfit-prone lags; BIC's smaller, more conservative lag is the")
    print("sturdier basis for a real claim.")


if __name__ == "__main__":
    main()

Loaded 101 days from /Users/nadia/Desktop/redditRun_june/q1_analysis/q1_monthly_series.csv

DESEASONALIZING (day-of-week + Patch Tuesday)
  Day-of-week + Patch-Tuesday deseasonalization:
    n_bat_ge1: R^2 = 0.0173 (little/no weekly pattern)
    n_bat_ge2: R^2 = 0.0137 (little/no weekly pattern)
    n_cve_run1: R^2 = 0.0599 (meaningful weekly pattern present)
    n_cve_run2: R^2 = 0.0315 (meaningful weekly pattern present)

n_bat_ge1  vs  n_cve_run1

  Stationarity check + common differencing order (on deseasonalized residuals):
    d=0: n_bat_ge1: ADF p=0.5551 (UNIT ROOT); linear trend check: TREND (slope=1.84220, p=2.655e-16)
           n_cve_run1: ADF p=0.3495 (UNIT ROOT); linear trend check: TREND (slope=-1.19724, p=1.025e-06)
    d=1: n_bat_ge1: ADF p=1.491e-09 (no unit root); linear trend check: no significant trend
           n_cve_run1: ADF p=3.311e-19 (no unit root); linear trend check: no significant trend
  Using d=1 for both series in this pair

  Selecting lag order (AIC/B

ValueError: maxlags is too large for the number of observations and the number of equations. The largest model cannot be estimated.